# Chapitre 3 - Exercices 3.26 à 3.30

| Exercice | Thème |
|---|---|
| 3.26 | Sélection ascendante et LAR : deux critères d'entrée différents |
| 3.27 | Conditions de Karush–Kuhn–Tucker du lasso, chemin linéaire par morceaux |
| 3.28 | Lasso avec une variable dupliquée |
| 3.29 | Ridge avec une variable dupliquée |
| 3.30 | L'elastic net comme un lasso sur des données augmentées |

Fonctions utilisées : `esl/ch03_regression_complements.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch03_regression_complements as rc

set_style()
np.set_printoptions(precision=4, suppress=True)

Les données du cancer de la prostate (échantillon d'apprentissage, 67 patients, 8
prédicteurs centrés-réduits) serviront d'exemple réel dans les exercices 3.27 à 3.29.

In [2]:
pro = datasets.load_prostate()
pro_tr = pro[pro["train"]]
pnames = list(pro.columns[:8])
Xp, _, _, _ = rc.standardize(pro_tr[pnames].to_numpy(float))
yp = pro_tr["lpsa"].to_numpy(float)
yp = yp - yp.mean()
print(Xp.shape)

(67, 8)


## Exercice 3.26 : Sélection ascendante contre LAR

**Énoncé (reformulé).** La sélection pas à pas ascendante (*forward stepwise*) fait entrer
à chaque étape la variable qui diminue le plus la somme des carrés résiduelle (RSS). Le LAR
fait progresser la variable de plus forte corrélation absolue avec le résidu courant.
Montrer que ces deux critères d'entrée ne sont pas nécessairement les mêmes.
*Indication :* notons $\mathbf x_{j\cdot\mathcal A}$ la variable $\mathbf x_j$ ajustée
linéairement sur les variables déjà dans le modèle ; montrer que le premier critère revient à
chercher le $j$ maximisant $|\operatorname{Corr}(\mathbf x_{j\cdot\mathcal A},\mathbf r)|$.

### Notations

Variables et réponse sont centrées ; les $\mathbf x_j$ sont de norme 1. Le modèle courant
contient les variables $\mathcal A$ ; $\mathbf H_{\mathcal A}$ est la projection orthogonale sur
$\operatorname{Vect}(\mathbf x_k,k\in\mathcal A)$ et $\mathbf r=\mathbf y-\mathbf H_{\mathcal A}\mathbf y$
le résidu (orthogonal aux variables actives). La variable ajustée est
$\mathbf x_{j\cdot\mathcal A}=\mathbf x_j-\mathbf H_{\mathcal A}\mathbf x_j$, orthogonale à
$\operatorname{Vect}(\mathbf x_k, k\in\mathcal A)$.

### Gain en RSS de la sélection ascendante

Ajouter $\mathbf x_j$ revient à projeter $\mathbf y$ sur
$\operatorname{Vect}(\mathbf x_k,k\in\mathcal A)\oplus\operatorname{Vect}(\mathbf x_{j\cdot\mathcal A})$
(même espace que avec $\mathbf x_j$, et somme **orthogonale**). La projection sur une somme
orthogonale est la somme des projections :

$$
\hat{\mathbf y}_{\mathcal A\cup\{j\}}=\mathbf H_{\mathcal A}\mathbf y+\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf y\rangle}{\|\mathbf x_{j\cdot\mathcal A}\|^2}\mathbf x_{j\cdot\mathcal A}.
$$

Par Pythagore, la RSS diminue de

$$
\Delta_j=\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf y\rangle^2}{\|\mathbf x_{j\cdot\mathcal A}\|^2}
=\frac{\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle^2}{\|\mathbf x_{j\cdot\mathcal A}\|^2}
=\|\mathbf r\|^2\cos^2(\mathbf x_{j\cdot\mathcal A},\mathbf r)
=\|\mathbf r\|^2\operatorname{Corr}^2(\mathbf x_{j\cdot\mathcal A},\mathbf r),
$$

où l'on a utilisé $\langle\mathbf x_{j\cdot\mathcal A},\mathbf H_{\mathcal A}\mathbf y\rangle=0$, et
le fait que pour des vecteurs centrés la corrélation est le cosinus de l'angle. Maximiser
$\Delta_j$ revient bien à maximiser $|\operatorname{Corr}(\mathbf x_{j\cdot\mathcal A},\mathbf r)|$.

### Critère du LAR

Le LAR choisit le $j$ maximisant $|\langle\mathbf x_j,\mathbf r\rangle|$. Comme $\mathbf r$ est
orthogonal aux variables actives, $\langle\mathbf x_j,\mathbf r\rangle=\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle$. En résumé, à résidu donné :

$$
\text{ascendante : }\max_j\frac{|\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle|}{\|\mathbf x_{j\cdot\mathcal A}\|},
\qquad
\text{LAR : }\max_j|\langle\mathbf x_{j\cdot\mathcal A},\mathbf r\rangle| .
$$

Les deux critères diffèrent par le facteur $1/\|\mathbf x_{j\cdot\mathcal A}\|\ge1$, d'autant plus
grand que $\mathbf x_j$ est **corrélée aux variables déjà présentes**. La sélection ascendante
favorise donc ces variables (une fois « nettoyées », elles peuvent expliquer beaucoup de
résidu), alors que le LAR les ignore. Les deux critères coïncident lorsque toutes les
$\|\mathbf x_{j\cdot\mathcal A}\|$ sont égales (par exemple si les prédicteurs sont orthogonaux,
ou à la première étape où $\mathcal A=\varnothing$).

### Un contre-exemple explicite

Dans une base orthonormée $(\mathbf q_1,\dots,\mathbf q_4)$ de vecteurs centrés de
$\mathbb R^N$ : $\mathbf x_1=\mathbf q_1$ (déjà dans le modèle),
$\mathbf x_2=\cos\theta\,\mathbf q_1+\sin\theta\,\mathbf q_2$ avec $\sin\theta=0{,}4$ (très corrélée à
$\mathbf x_1$), $\mathbf x_3=\mathbf q_3$, et le résidu
$\mathbf r=0{,}75\,\mathbf q_2+0{,}5\,\mathbf q_3+0{,}2\,\mathbf q_4$. Alors
$\langle\mathbf x_2,\mathbf r\rangle=0{,}3<\langle\mathbf x_3,\mathbf r\rangle=0{,}5$ (le LAR choisit
$\mathbf x_3$), mais $\|\mathbf x_{2\cdot\mathcal A}\|=0{,}4$ donne $\Delta_2=0{,}3^2/0{,}16=0{,}5625>\Delta_3=0{,}25$
(la sélection ascendante choisit $\mathbf x_2$).

In [3]:
rng = np.random.RandomState(0)
N = 50
Qfull = np.linalg.qr(np.column_stack([np.ones(N), rng.randn(N, 4)]))[0]
q = Qfull[:, 1:]                                  # 4 vecteurs centrés orthonormés
sin_t = 0.4
x1, x2, x3 = q[:, 0], np.sqrt(1 - sin_t ** 2) * q[:, 0] + sin_t * q[:, 1], q[:, 2]
r = 0.75 * q[:, 1] + 0.5 * q[:, 2] + 0.2 * q[:, 3]
H = np.outer(x1, x1)                              # projection sur Vect(x1)
for name, x in [("x2", x2), ("x3", x3)]:
    xa = x - H @ x
    print(f"{name} : <x, r> = {x @ r:.3f}   ||x.A|| = {np.linalg.norm(xa):.3f}"
          f"   baisse de RSS = {(xa @ r) ** 2 / (xa @ xa):.4f}")

x2 : <x, r> = 0.300   ||x.A|| = 0.400   baisse de RSS = 0.5625
x3 : <x, r> = 0.500   ||x.A|| = 1.000   baisse de RSS = 0.2500


### Sur des données simulées : les ordres d'entrée diffèrent souvent

On compare l'ordre d'entrée des variables de la sélection ascendante et du LAR (chacun avec
son propre résidu) sur 300 jeux simulés à prédicteurs corrélés.

In [4]:
rng = np.random.RandomState(1)
p = 6
cov = 0.8 ** np.abs(np.subtract.outer(np.arange(p), np.arange(p)))
L = np.linalg.cholesky(cov)
first_diff = []
for _ in range(300):
    Xs = rng.randn(60, p) @ L.T
    ys = Xs @ rng.randn(p) + 2 * rng.randn(60)
    Xs, _, _, _ = rc.standardize(Xs, unit_norm=True)
    ys -= ys.mean()
    order_fs = rc.forward_stepwise(Xs, ys)[0]
    order_lar = [j for ev, j in rc.lars(Xs, ys)["events"] if ev == "entrée"]
    diff = [k for k in range(p) if order_fs[k] != order_lar[k]]
    first_diff.append(diff[0] + 1 if diff else 0)
first_diff = np.array(first_diff)
print("proportion de jeux où les ordres diffèrent :", np.mean(first_diff > 0))

proportion de jeux où les ordres diffèrent : 0.8933333333333333


In [ ]:
fig, ax = plt.subplots()
counts = [np.sum(first_diff == k) for k in range(1, p + 1)]
ax.bar(range(1, p + 1), counts, color=BLUE)
ax.set_xlabel("première position où les ordres d'entrée diffèrent")
ax.set_ylabel("nombre de jeux simulés (sur 300)")
ax.set_title("Sélection ascendante et LAR : ordres d'entrée")
plt.show()

Les ordres coïncident toujours à la première position (critères identiques quand
$\mathcal A=\varnothing$), puis divergent fréquemment dès la deuxième.

> **À retenir.** La sélection ascendante choisit la variable dont la **partie nouvelle**
> $\mathbf x_{j\cdot\mathcal A}$ est la plus corrélée au résidu ; le LAR celle dont la variable
> **brute** a le plus grand produit scalaire avec le résidu. Les deux diffèrent par le facteur
> $1/\|\mathbf x_{j\cdot\mathcal A}\|$, qui avantage en sélection ascendante les variables
> corrélées à celles déjà présentes.

## Exercice 3.27 : Lasso et LAR : conditions de Karush–Kuhn–Tucker

**Énoncé (reformulé).** On considère le lasso sous forme lagrangienne : avec
$L(\beta)=\frac12\sum_i(y_i-\sum_jx_{ij}\beta_j)^2$, on minimise
$L(\beta)+\lambda\sum_j|\beta_j|$ (éq. (3.88)) pour un $\lambda>0$ fixé.

(a) En écrivant $\beta_j=\beta_j^+-\beta_j^-$ avec $\beta_j^\pm\ge0$, le critère devient
$L(\beta)+\lambda\sum_j(\beta_j^++\beta_j^-)$. Montrer que le lagrangien associé est
$L(\beta)+\lambda\sum_j(\beta_j^++\beta_j^-)-\sum_j\lambda_j^+\beta_j^+-\sum_j\lambda_j^-\beta_j^-$ (éq. (3.89))
et que les conditions de Karush–Kuhn–Tucker (KKT) sont
$\nabla L(\beta)_j+\lambda-\lambda_j^+=0$, $-\nabla L(\beta)_j+\lambda-\lambda_j^-=0$,
$\lambda_j^+\beta_j^+=0$, $\lambda_j^-\beta_j^-=0$, avec positivité de tous les paramètres et
multiplicateurs.

(b) Montrer que $|\nabla L(\beta)_j|\le\lambda$ pour tout $j$, et que les conditions KKT
entraînent l'un des trois scénarios : $\lambda=0\Rightarrow\nabla L(\beta)_j=0$ ;
$\beta_j^+>0,\lambda>0\Rightarrow\lambda_j^+=0,\ \nabla L(\beta)_j=-\lambda,\ \beta_j^-=0$ ;
$\beta_j^->0,\lambda>0\Rightarrow\lambda_j^-=0,\ \nabla L(\beta)_j=\lambda,\ \beta_j^+=0$.
En déduire que pour un prédicteur « actif » ($\beta_j\ne0$),
$\nabla L(\beta)_j=-\lambda\operatorname{sign}(\beta_j)$, et relier $\lambda$ à la corrélation
entre $\mathbf x_j$ et le résidu.

(c) Si l'ensemble des prédicteurs actifs ne change pas pour $\lambda_0\ge\lambda\ge\lambda_1$,
montrer qu'il existe un vecteur $\gamma_0$ tel que
$\hat\beta(\lambda)=\hat\beta(\lambda_0)-(\lambda-\lambda_0)\gamma_0$ (éq. (3.90)) : le chemin
du lasso est linéaire entre $\lambda_0$ et $\lambda_1$.